In [0]:
df = spark.table("samples.nyctaxi.trips")
display(df.limit(10))

In [0]:
trip_counts_df = spark.sql("""
SELECT 
  CAST(tpep_pickup_datetime AS DATE) AS pickup_date, 
  COUNT(*) AS trip_count 
FROM nyctaxi_trips 
GROUP BY CAST(tpep_pickup_datetime AS DATE) 
ORDER BY pickup_date
""")

trip_counts_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/my_files/nyctaxi_trip_counts_delta")

In [0]:
df1 = spark.read.csv("/Volumes/workspace/default/my_files/strait_of_hormuz_shipping_disruption_2026 (1).csv", header=True, inferSchema=True)

 

Day 2 (Tomorrow):

Learn Delta Lake: write/read Delta tables, schema evolution, MERGE INTO

Build one notebook that does upserts

Document everything in your repo README

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("age", IntegerType(), True),
])

# Initial data for the target table
data_target = [
    (1, "Alice", "Anders", 30),
    (2, "Bob",   "Bauer",  40),
    (3, "Chris", "Conrad", 50),
]

df_target = spark.createDataFrame(data_target, schema)

# Write as a managed Delta table (adjust catalog.schema if needed)
df_target.write.format("delta").mode("overwrite").saveAsTable("default.people_demo")

display(spark.table("default.people_demo"))

In [0]:
data_updates = [
    (2, "Bob",  "Bauer",  41),   # existing id, age changed
    (4, "Dora", "Diehl",  28),   # new id, should be inserted
]

df_updates = spark.createDataFrame(data_updates, schema)

# Optionally: create temp view, as docs often do
df_updates.createOrReplaceTempView("people_demo_updates")

display(df_updates)

In [0]:
delta_table = DeltaTable.forName(spark, "default.people_demo")

(delta_table.alias("t")
    .merge(
        df_updates.alias("s"),
        "t.id = s.id"          # join condition
    )
    .whenMatchedUpdateAll()    # update all columns when ids match
    .whenNotMatchedInsertAll() # insert new rows when id not found
    .execute()
)

In [0]:
display(spark.table("default.people_demo").orderBy("id"))